# 00 — Ground truth

Why this repo builds its own environments instead of only using Gymnasium.

Every finite environment here defines its dynamics **once**, as a transition table `P`:

```
P[s][a] -> [(probability, next_state, reward, terminated), ...]
```

`step()` samples from that table. `envs.solvers` reads the same table. So the exact Q function
cannot drift away from the environment the agent is actually interacting with, and when a learned
Q disagrees with `value_iteration`, **the agent is wrong** — never the comparison.

No standard benchmark gives you that, and it changes what debugging feels like.

In [ ]:
import numpy as np

from diagnostics.render import compare_grids, policy_overlay, value_heatmap, visit_heatmap
from envs import solvers
from envs.gridworld import GridWorld

GAMMA = 0.95

## The exact answer, for free

No training, no sampling, no seeds. This is what optimal *is*.

In [ ]:
env = GridWorld(size=7, reward_mode="sparse", walls=[(2, 2), (2, 3), (3, 2), (4, 5)])

print(policy_overlay(env, env.true_q(GAMMA), "Q* with greedy arrows"))
print(f"\nexact return from the start state: {env.optimal_return(GAMMA):.4f}")

## Scoring a policy without sampling

`policy_return` computes the expected discounted return exactly. It has **no variance**, so a
difference between two policies is a real difference and not seed noise.

This is the tool that makes "is A better than B" answerable in one line instead of thirty seeds.

In [ ]:
optimal = env.optimal_policy(GAMMA)
rng = np.random.default_rng(0)

print(f"optimal policy          {env.policy_return(optimal, GAMMA):+.4f}")
for epsilon in (0.0, 0.1, 0.3, 1.0):
    pi = solvers.epsilon_greedy_policy(env.true_q(GAMMA), epsilon)
    print(f"epsilon-greedy eps={epsilon:<4}  {env.policy_return(pi, GAMMA):+.4f}")

random_policy = rng.integers(0, 4, env.n_states)
print(f"a random policy         {env.policy_return(random_policy, GAMMA):+.4f}")

## Bug #1, and why the tabular version of it does nothing

Forgetting to mask terminal transitions is the most common bug in deep RL. Here is value iteration
with the mask deleted — everything else identical.

In [ ]:
def unmasked_backup(model, n_states, n_actions, gamma, sweeps=2000):
    V = np.zeros(n_states)
    for _ in range(sweeps):
        backup = model.reward + gamma * V[model.next_state]  # <- no `* model.continues`
        Q = np.bincount(
            model.sa, weights=model.prob * backup, minlength=n_states * n_actions
        ).reshape(n_states, n_actions)
        V = Q.max(axis=1)
    return V


truth = env.true_v(GAMMA)
naive = unmasked_backup(
    solvers.flatten(env.P, env.n_states, env.n_actions), env.n_states, env.n_actions, GAMMA
)
print(f"max |error| with the mask deleted: {np.abs(naive - truth).max():.6f}")

Zero. Nothing happened.

In this table the goal is absorbing with zero reward, so `V(goal)` is 0 whether you mask or not,
and `gamma * 0` is 0 either way. **The mask only matters when the value after termination is
non-zero** — which is exactly what a neural network gives you, since it will happily predict some
arbitrary value for a state the episode never continues from.

Worth knowing before trusting any tabular test of this bug.

## The same bug in its realistic form

A vectorised environment resets on your behalf, so the observation after a terminal step already
belongs to the **next** episode. Store that as `next_obs`, forget the mask, and reaching the goal
bootstraps the value of starting again.

In [ ]:
def auto_reset_model(env):
    start = int(np.argmax(env.start_distribution))
    P = {
        s: {
            a: [
                (prob, start if terminated else s2, reward, terminated)
                for prob, s2, reward, terminated in env.P[s][a]
            ]
            for a in range(env.n_actions)
        }
        for s in range(env.n_states)
    }
    return solvers.flatten(P, env.n_states, env.n_actions)


broken = unmasked_backup(auto_reset_model(env), env.n_states, env.n_actions, GAMMA)
print(compare_grids(env, broken, truth, gamma_label=f"(gamma={GAMMA})"))

Every error the same sign, growing toward the goal. **That shape is a systematic bias**, and no
learning rate will fix it.

Compare against what an under-trained but correct agent looks like.

In [ ]:
noisy = truth + rng.normal(0, 0.05, env.n_states)
print(compare_grids(env, noisy, truth, gamma_label=f"(gamma={GAMMA})"))

Speckled, both signs, no structure. That agent needs more data. The one above needs a different
line of code. **Both produce the same flat reward curve.**

## Half of "it is not learning" is "it never went there"

In [ ]:
counts = np.zeros(env.n_states)
for episode in range(400):
    env.reset(seed=episode)
    counts[env.state] += 1
    while True:
        _, _, terminated, truncated, _ = env.step(int(rng.integers(0, 4)))
        counts[env.state] += 1
        if terminated or truncated:
            break

print(visit_heatmap(env, counts, "visits under a uniform random policy"))
print(f"\ngoal reached {int(counts[env.index(env.goal)])} times in 400 episodes")

## Reward hacking, and why it depends on the discount

`misspecified` mode gives a **one-sided** progress bonus: reward for moving toward the goal, no
penalty for moving away. The difference from correct potential-based shaping is one `max(0, ...)`.

Farming the bonus is worth `b / (1 - gamma^2)`. Reaching the goal is worth `1 + b` and then stops.
So which one wins depends on the discount — the same reward bug is harmless at one horizon and
catastrophic at another.

In [ ]:
hacked = GridWorld(size=6, reward_mode="misspecified")


def reaches_goal(env, gamma, max_steps=500):
    pi = env.optimal_policy(gamma)
    env.reset(seed=0)
    for _ in range(max_steps):
        _, _, terminated, truncated, _ = env.step(int(pi[env.state]))
        if terminated:
            return True
        if truncated:
            return False
    return False


for gamma in (0.80, 0.85, 0.90, 0.95, 0.99):
    verdict = "reaches the goal" if reaches_goal(hacked, gamma) else "HACKS: never enters the goal"
    print(f"gamma={gamma:<5}  {verdict}")

print()
print(hacked.render_policy(hacked.optimal_policy(0.95)))
print("\nnote the arrows next to the goal pointing away from it")

The agent is not broken. It is doing exactly what it was asked to do.

---

**Next:** algorithms, tier by tier. See `docs/rl-curriculum-plan.md`.